In [24]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import pyarrow.parquet as pq

DATA_ROOT = "/media/mikhail/Data/research/dl_momentum/project_data/data/datasets"

In [25]:
def get_ts_dt(str_timestamps):
    return pd.to_datetime(str_timestamps, utc=True)

def get_times(interval):
    return (
        pd.Timestamp(interval[0], tz="UTC").value,
        pd.Timestamp(interval[1], tz="UTC").value,
    )

def get_sliced_df(df, times):
    return df.loc[(df.timestamp >= times[0]) & (df.timestamp <= times[1])]

In [42]:
TRAIN_INTERVAL = ("2000-01-01", "2016-01-01")

tmp_times = get_times(TRAIN_INTERVAL)
df_data = pq.read_table(f"{DATA_ROOT}/top3000_data_df.parquet").to_pandas()
df_data["timestamp"] = pd.to_datetime(df_data.index, utc=True).astype(int)
# df_data = df_data.dropna(axis=1, how="all")
df_data = get_sliced_df(df_data, tmp_times)
 
# df_dividends = pq.read_table(f"{DATA_ROOT}/top3000_dividends.parquet").to_pandas()

# df_market_caps = pq.read_table(f"{DATA_ROOT}/top3000_market_caps.parquet").to_pandas()
# df_market_caps["timestamp"] = pd.to_datetime(df_market_caps.index, utc=True).astype(int)
# df_market_caps = get_sliced_df(df_market_caps, tmp_times)

df_presence = pq.read_table(f"{DATA_ROOT}/top3000_presence_matrix.parquet").to_pandas()
df_presence["timestamp"] = pd.to_datetime(df_presence.index, utc=True).astype(int)
# df_presence = df_presence.dropna(axis=1, how="all")
df_presence = get_sliced_df(df_presence, tmp_times)

In [8]:
fig = go.Figure()
ts_dt = get_ts_dt(df_data.timestamp)

col_subset = df_data.columns[:10]

for col in col_subset:
    fig.add_trace(go.Scatter(x=ts_dt, y=df_data[col], name=col, line=dict(shape="hv")))

fig.show(renderer="browser")

In [ ]:
# we can clusterize by correlation of returns for the past X days, for example and trade momentum on clusters, not for each asset

In [ ]:
# why to use contrastive learning ?
# we can achieve non-linear clustering of correlations 

In [ ]:
# then try to buy\sell whole clusters 

In [41]:
df_presence

pmpid,10012,10025,10026,10032,10034,10035,10048,10051,10064,10071,...,93420,93422,93423,93426,93428,93429,93430,93433,93436,timestamp
date,,,,,,,,,,,,,,,,,,,,,
2000-01-01,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,946684800000000000
2000-01-02,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,946771200000000000
2000-01-03,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,946857600000000000
2000-01-04,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,946944000000000000
2000-01-05,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,947030400000000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-12-28,NaN,1.0,1.0,1.0,NaN,NaN,NaN,1.0,NaN,NaN,...,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,1.0,1451260800000000000
2015-12-29,NaN,1.0,1.0,1.0,NaN,NaN,NaN,1.0,NaN,NaN,...,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,1.0,1451347200000000000
2015-12-30,NaN,1.0,1.0,1.0,NaN,NaN,NaN,1.0,NaN,NaN,...,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,1.0,1451433600000000000


In [38]:
df_data.shape

(4025, 10612)

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from itertools import combinations

ALPHA = 0.5  # EMA weight on previous rolling corr vs new signal
THRESHOLD = 0.7  # Minimum correlation for an edge to exist
MIN_OBS = 3  # Min non-NaN observations per instrument to include in corr
WEEK_FREQ = "W-FRI"

# Instrument IDs are integer-like; factor columns (seasonality, size, etc.) are not
instrument_cols = [c for c in df_presence.columns if str(c).lstrip("-").isdigit()]

df_presence.index = pd.to_datetime(df_presence.index)
df_data.index = pd.to_datetime(df_data.index)

prices = df_data[instrument_cols]
log_ret = np.log(prices / prices.shift(1))  # (days × instruments)

ret_by_week = {we: grp for we, grp in log_ret.resample(WEEK_FREQ) if not grp.empty}
pres_by_week = {
    we: grp
    for we, grp in df_presence[instrument_cols].resample(WEEK_FREQ)
    if not grp.empty
}

week_ends = sorted(ret_by_week.keys())  # ordered list of week-end timestamps

rolling_corr = pd.DataFrame(
    np.eye(len(instrument_cols)),
    index=instrument_cols,
    columns=instrument_cols,
    dtype=np.float64,
)


def get_present(week_end: pd.Timestamp) -> list[int]:
    """Instruments with presence == 1.0 on ALL trading days of the week."""
    grp = pres_by_week.get(week_end)
    if grp is None or grp.empty:
        return []
    mask = (grp == 1.0).all(axis=0)
    return mask.index[mask].tolist()


def week_corr(week_end: pd.Timestamp, instruments: list) -> pd.DataFrame | None:
    grp = ret_by_week.get(week_end)
    if grp is None:
        return None
    sub = grp[instruments].copy()
    # Drop instruments with too few observations
    valid = [c for c in sub.columns if sub[c].notna().sum() >= MIN_OBS]
    if len(valid) < 2:
        return None
    return sub[valid].corr()


results = []

for i, week_end in enumerate(week_ends):
    if i < 4:  # need at least 4 weeks of history
        continue

    present = get_present(week_end)
    if len(present) < 2:
        continue

    corr_sum = pd.DataFrame(0.0, index=present, columns=present)

    for lag in range(1, 5):
        c = week_corr(week_ends[i - lag], present)
        if c is None:
            continue
        # Reindex to full `present` set; missing pairs → 0
        c_aligned = c.reindex(index=present, columns=present).fillna(0.0)
        corr_sum = corr_sum.add(c_aligned)

    sub = rolling_corr.loc[present, present]
    rolling_corr.loc[present, present] = sub * ALPHA + (1 - ALPHA) * corr_sum

    G = nx.Graph()
    G.add_nodes_from(present)

    corr_sub = rolling_corr.loc[present, present]
    for a, b in combinations(present, 2):
        w = corr_sub.loc[a, b]
        if w >= THRESHOLD:
            G.add_edge(a, b, weight=float(w))

    cliques = [c for c in nx.find_cliques(G) if len(c) >= 2]

    results.append(
        {
            "week_end": week_end,
            "n_present": len(present),
            "present": present,
            "cliques": cliques,  # list of lists of instrument IDs
            "n_cliques": len(cliques),
            "corr_matrix": corr_sub.copy(),  # snapshot of EMA corr for present instruments
        }
    )

print(f"Weeks processed : {len(results)}")
print(f"Last week       : {results[-1]['week_end'].date()}")
print(f"Cliques found   : {results[-1]['n_cliques']}")
print(f"Example cliques : {results[-1]['cliques'][:5]}")